# Tiny LLM — Optuna Hyperparameter Search

Runs short training trials using the real `GPTModel` + `MemmapDataset` from `src/tiny_llm`.
Each trial samples a configuration, trains for a fixed step budget, and reports validation loss.
Optuna's TPE sampler focuses subsequent trials on the most promising regions of the search space.

---

## What we already know — summary of runs_final

These findings from previous ablation runs motivate the search space below.

### Architecture shape (Phase 1 — 1 000 steps, eff. batch 65 K tok/step)
All shapes use d_model=384. Shape C (L=12, d_ff=1024) narrowly wins across all seeds.

| Shape | n_layers | d_ff | avg val loss |
|-------|----------|------|--------------|
| A | 10 | 1024 | 4.884 |
| B | 12 | 896  | 4.885 |
| **C** | **12** | **1024** | **4.875** ← winner |
| D (d=320) | 16 | 853  | 4.946 |

### Phase 1.5A — local search around Shape C (same budget, eff. batch 65 K tok/step)
Wide-shallow edges out deep configs at this step budget.

| Variant | n_layers | d_ff | avg val loss |
|---------|----------|------|--------------|
| **wide_shallow** | **10** | **1229** | **4.868** ← winner |
| deep_balanced | 14 | 878  | 4.878 |
| very_deep_lean | 16 | 768  | 4.891 |

### Phase 1.5B — d_model / d_ff / n_layers sweeps (1 000 steps, **eff. batch 262 K tok/step**)
4× larger effective batch reveals much stronger signal. d_model=448 is clearly better.
These are the most informative results.

| Config | d_model | n_layers | d_ff | n_params | val loss |
|--------|---------|----------|------|----------|----------|
| d=320 baseline | 320 | 12 | 1024 | 32.8 M | 4.549 |
| ff=896  | 384 | 12 | 896  | 38.8 M | 4.494 |
| **C baseline** | **384** | **12** | **1024** | **40.5 M** | **4.467** |
| ff=1152 | 384 | 12 | 1152 | 42.3 M | 4.444 |
| L=10    | 384 | 10 | 1024 | 37.0 M | 4.481 |
| L=14    | 384 | 14 | 1024 | 44.1 M | 4.444 |
| **d=448** | **448** | **12** | **1024** | **48.7 M** | **4.366** ← best |

### Phase 2 — learning rate sweep (1 500 steps, Shape C, eff. batch 262 K tok/step)
Higher LR wins convincingly. The optimum is likely above 1e-3.

| lr | val loss | val PPL |
|----|----------|---------|
| 3e-4 | 4.553 | 94.9 |
| 6e-4 | 4.181 | 65.4 |
| **1e-3** | **3.998** | **54.5** ← best |

### Key conclusions for this search
- **d_model=448** beats 384 — should explore further (only one seed/LR tested)
- **n_layers=12–14** sweet spot; 16 layers not worth the cost at this param budget
- **d_ff** scale matters; 1024–1280 is the target zone
- **lr≥1e-3** is necessary; explore 1e-3 to 3e-3
- **dropout=0.0** is settled (P0: dropout hurts)
- **batch=64, grad_accum=4** (262 K tokens/step) beats smaller effective batches

In [ ]:
# Uncomment if needed:
# %pip install -q optuna optuna-dashboard plotly pandas

In [ ]:
# ── Environment setup (local + Google Colab) ─────────────────────────────────
import os
import sys
import getpass
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # Clone repo (only once)
    if not Path("40m-tiny-llm").exists():
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token: ")
        subprocess.run(
            ["git", "clone", f"https://{token}@github.com/marcbloech/40m-tiny-llm.git"],
            check=True,
        )
    os.chdir("40m-tiny-llm")

    # Install dependencies
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "optuna", "optuna-dashboard"],
        check=True,
    )

# ── Repo root + import path ──────────────────────────────────────────────────
# Works whether you run from the repo root or from notebooks/
_cwd = Path(os.getcwd()).resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "pyproject.toml").exists() and (p / "src").exists()),
    _cwd,
)
os.chdir(REPO_ROOT)

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Data paths ───────────────────────────────────────────────────────────────
if IN_COLAB:
    import shutil
    # Expects tokenized data in Google Drive at MyDrive/tiny_llm/data/tokenized/
    GDRIVE_TOK = Path("/content/drive/MyDrive/tiny_llm/data/tokenized")
    DATA_DIR   = REPO_ROOT / "data" / "tokenized"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for fname in ["train.bin", "val.bin"]:
        dst = DATA_DIR / fname
        src = GDRIVE_TOK / fname
        if not dst.exists():
            if src.exists():
                print(f"Copying {fname} from Drive ({src.stat().st_size / 1e9:.2f} GB)...")
                shutil.copy(src, dst)
            else:
                raise FileNotFoundError(
                    f"{fname} not found on Drive at {src}.\n"
                    "Run `python main.py --stage data` locally first, "
                    "then upload data/tokenized/ to MyDrive/tiny_llm/data/tokenized/."
                )
else:
    DATA_DIR = REPO_ROOT / "data" / "tokenized"

TRAIN_BIN = DATA_DIR / "train.bin"
VAL_BIN   = DATA_DIR / "val.bin"

assert TRAIN_BIN.exists(), f"Missing: {TRAIN_BIN}"
assert VAL_BIN.exists(),   f"Missing: {VAL_BIN}"
print(f"Repo root : {REPO_ROOT}")
print(f"train.bin : {TRAIN_BIN.stat().st_size / 1e9:.2f} GB")
print(f"val.bin   : {VAL_BIN.stat().st_size / 1e9:.2f} GB")

In [ ]:
import gc
import math
import random

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tiny_llm.model.transformer import GPTModel
from tiny_llm.data.dataset import MemmapDataset
from tiny_llm.training.optimizer import create_optimizer, create_scheduler

print("tiny_llm imported OK")

In [ ]:
import os
# Reduces cross-trial memory fragmentation on CUDA
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"  # AMP only on CUDA

print(f"device : {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")
print(f"AMP    : {USE_AMP}")

In [ ]:
# ── Device-aware batch sizing ─────────────────────────────────────────────────
# Target: ~262K tokens/step — the effective batch from P15B that gave the clearest signal.
# Batch size is NOT a search parameter; it is fixed per hardware tier so that
# trial comparisons reflect architecture differences, not batch-size effects.

if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb >= 35:        # A100 40/80 GB (Colab Pro+, academic)
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 32, 8, 32
    elif vram_gb >= 14:      # T4 16 GB (Colab free)
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 16, 16, 32
    else:                    # K80 12 GB or smaller
        BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 8, 32, 16
elif device.type == "mps":   # Apple Silicon
    BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 16, 16, 16
else:                        # CPU (very slow, for debugging only)
    BATCH_SIZE, GRAD_ACCUM, EVAL_BATCH_SIZE = 4, 4, 4

EFF_TOK = BATCH_SIZE * GRAD_ACCUM * 1024
print(
    f"Batch config: {BATCH_SIZE} seqs × {GRAD_ACCUM} grad_accum "
    f"= {BATCH_SIZE * GRAD_ACCUM} seqs/step → {EFF_TOK / 1e3:.0f}K tokens/step"
)

In [ ]:
# ── Trial budget ──────────────────────────────────────────────────────────────
# Each Optuna trial trains the model for TRAIN_STEPS optimizer steps.
# Architecture rankings stabilise within 5–10 % of full compute (Hoffmann et al., 2022).
#
# Rough timing on Colab T4 (batch=16, grad_accum=16, d_model=384):
#   TRAIN_STEPS=300 → ~5 min/trial  →  20 trials ≈ 1.5 h
#   TRAIN_STEPS=500 → ~9 min/trial  →  20 trials ≈ 3 h

TRAIN_STEPS  = 500   # optimizer steps per trial
WARMUP_STEPS = 50    # LR warmup (10 % of TRAIN_STEPS)
EVAL_STEPS   = 40    # val batches averaged for each loss estimate
REPORT_EVERY = 100   # intermediate report interval (pruner decision points)
N_TRIALS     = 20    # total Optuna trials to run

# ── Study persistence ─────────────────────────────────────────────────────────
# In Colab the DB lives on Google Drive so it survives session restarts.
# Locally it sits next to this notebook.
if IN_COLAB:
    GDRIVE_OPTUNA = Path("/content/drive/MyDrive/tiny_llm/optuna")
    GDRIVE_OPTUNA.mkdir(parents=True, exist_ok=True)
    DB_PATH = GDRIVE_OPTUNA / "tiny_llm_optuna_search.db"
else:
    DB_PATH = REPO_ROOT / "notebooks" / "tiny_llm_optuna_search.db"
STORAGE    = f"sqlite:///{DB_PATH}"
STUDY_NAME = "tiny_llm_arch_search"

# ── Fixed model constants (settled by ablation experiments — not swept here) ──
VOCAB_SIZE     = 50257   # GPT-2 tokenizer
CONTEXT_LENGTH = 1024
ACTIVATION     = "swiglu"   # P0: swiglu > gelu
POS_ENC        = "rope"     # baseline across all phases
NORM_POS       = "pre"      # baseline across all phases
WEIGHT_TYING   = True       # saves V×d params; always on
BIAS           = False      # modern default
QK_NORM        = False      # P0: marginal gain, not used in final
DROPOUT        = 0.0        # P0: dropout hurts at this scale

print(f"Trials        : {N_TRIALS} × {TRAIN_STEPS} steps")
print(f"Tokens/trial  : {TRAIN_STEPS * EFF_TOK / 1e6:.1f} M")
print(f"DB            : {DB_PATH}")

## Search space

Seven parameters are swept. Everything else is fixed based on ablation evidence.

| Parameter | Range | Scale | Rationale |
|-----------|-------|-------|-----------|
| `d_model` | {384, 448} | categorical | 384 = Shape C baseline; 448 = best in P1.5B (needs >1 seed confirmation) |
| `n_layers` | 8 – 14 | int | Reduced from 10–16; keeps all configs within 40 M param budget |
| `d_ff` | 640 – 1 280, step 64 | int | Lower bound reduced to 640 to allow d_model=448 to stay under budget |
| `activation` | {gelu, swiglu} | categorical | P0 SwiGLU vs GELU test was at 7 M params, single seed — needs confirmation at scale |
| `norm_position` | {pre, post, pre_post} | categorical | Phase 3 norm ablations were never run — literature favours pre-norm but empirical test is missing |
| `lr` | 2e-4 – 5e-3 | log | P2 showed lr had the **greatest effect** of any parameter; wider range ensures we find the true peak |
| `weight_decay` | 0.05 – 0.30 | linear | 0.1 used throughout but never swept; standard AdamW range |

**Derived (not searched):**
- `n_heads = d_model // 32` — keeps head_dim = 32, consistent with all experiments
- `positional_encoding = "rope"`, `weight_tying = True`, `bias = False`, `qk_norm = False` — fixed from ablation winners
- Configs exceeding **42 M parameters** are pruned immediately (40 M target, ~5 % tolerance)
- `approx_param_count` is activation-aware: SwiGLU counts 3 FFN matrices, GELU counts 2


In [ ]:
def approx_param_count(d_model: int, n_layers: int, d_ff: int, activation: str = "swiglu") -> int:
    """Estimate total parameters (weight-tied output projection).

    SwiGLU uses 3 FFN weight matrices (gate, up, down);
    GELU uses 2 (fc1, fc2).  The count differs for the same d_ff.
    """
    ffn_matrices = 3 if activation == "swiglu" else 2
    emb       = VOCAB_SIZE * d_model          # token embedding (shared with lm_head)
    per_block = (4 * d_model**2              # QKV + out_proj
                 + ffn_matrices * d_model * d_ff  # FFN weights
                 + 2 * d_model)              # 2 × RMSNorm scale vectors
    return emb + n_layers * per_block + d_model  # final RMSNorm


def _next_batch(iterator, loader):
    """Advance iterator, resetting when the dataset is exhausted."""
    try:
        return next(iterator)
    except StopIteration:
        return next(iter(loader))


@torch.no_grad()
def evaluate(model: nn.Module, val_loader: DataLoader, n_steps: int) -> float:
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_loader):
        if i >= n_steps:
            break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device.type, enabled=USE_AMP):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


# Pre-build datasets once — MemmapDataset is lazy (mmap), cheap to instantiate.
train_ds = MemmapDataset(TRAIN_BIN, context_length=CONTEXT_LENGTH)
val_ds   = MemmapDataset(VAL_BIN,   context_length=CONTEXT_LENGTH)
print(f"train samples: {len(train_ds):,}  |  val samples: {len(val_ds):,}")
# Build loaders once — reused across all trials (avoids pinned-memory churn).
# pin_memory=False: pinned allocations from recreated loaders accumulate and
#                   cause OOM across trials even after del model + empty_cache().
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False, drop_last=False,
)
print(f"train loader: {len(train_loader):,} batches  |  val loader: {len(val_loader):,} batches")


In [ ]:
def objective(trial: optuna.Trial) -> float:
    # ── 1. Sample hyperparameters ─────────────────────────────────────────────
    d_model     = trial.suggest_categorical("d_model",      [384, 448])
    n_heads     = d_model // 32  # head_dim=32 throughout
    trial.set_user_attr("n_heads", n_heads)
    n_layers    = trial.suggest_int("n_layers",  8, 14)
    d_ff        = trial.suggest_int("d_ff",      640, 1280, step=64)
    activation  = trial.suggest_categorical("activation",   ["gelu", "swiglu"])
    norm_pos    = trial.suggest_categorical("norm_position", ["pre", "post", "pre_post"])
    lr           = trial.suggest_float("lr",           2e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.05, 0.30)

    # ── 2. Early-prune oversized configs ─────────────────────────────────────
    # approx_param_count is activation-aware: SwiGLU=3 FFN matrices, GELU=2.
    # Hard budget: 42 M (40 M target + small tolerance).
    n_params = approx_param_count(d_model, n_layers, d_ff, activation)
    trial.set_user_attr("n_params", n_params)
    trial.set_user_attr("n_params_M", round(n_params / 1e6, 1))
    if n_params > 42_000_000:
        raise optuna.TrialPruned()

    # model is declared here so the finally block can always reference it
    model = None

    try:
        # ── 3. Build model ────────────────────────────────────────────────────
        # Inside try: .to(device) is a CUDA call and can raise AcceleratorError
        # if a previous trial left the CUDA context in a bad state.
        model = GPTModel({
            "vocab_size":          VOCAB_SIZE,
            "context_length":      CONTEXT_LENGTH,
            "n_layers":            n_layers,
            "n_heads":             n_heads,
            "d_model":             d_model,
            "d_ff":                d_ff,
            "dropout":             DROPOUT,
            "activation":          activation,
            "norm_position":       norm_pos,
            "weight_tying":        WEIGHT_TYING,
            "bias":                BIAS,
            "qk_norm":             QK_NORM,
            "positional_encoding": POS_ENC,
            "rope_theta":          10_000.0,
        }).to(device)

        # ── 4. Optimizer + LR scheduler ───────────────────────────────────────
        optimizer = create_optimizer(
            model, optimizer_name="adamw",
            learning_rate=lr, weight_decay=weight_decay, betas=(0.9, 0.95),
        )
        scheduler = create_scheduler(
            optimizer, lr_schedule="cosine",
            warmup_steps=WARMUP_STEPS, max_steps=TRAIN_STEPS, min_lr_ratio=0.1,
        )
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

        # ── 5. Training loop ──────────────────────────────────────────────────
        model.train()
        train_iter = iter(train_loader)
        optimizer.zero_grad(set_to_none=True)

        for step in range(TRAIN_STEPS):
            for _ in range(GRAD_ACCUM):
                x, y = _next_batch(train_iter, train_loader)
                x, y = x.to(device), y.to(device)
                with torch.amp.autocast(device.type, enabled=USE_AMP):
                    _, loss = model(x, y)
                scaler.scale(loss / GRAD_ACCUM).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            if (step + 1) % REPORT_EVERY == 0:
                val_loss = evaluate(model, val_loader, EVAL_STEPS)
                trial.report(val_loss, step)
                if trial.should_prune():
                    raise optuna.TrialPruned()

        # ── 6. Final evaluation ───────────────────────────────────────────────
        return evaluate(model, val_loader, EVAL_STEPS)

    except optuna.TrialPruned:
        raise  # pass through — Optuna handles this normally

    except Exception as e:
        # Catch ALL exceptions here, not just RuntimeError.
        # torch.AcceleratorError (PyTorch 2.4+) does not inherit from RuntimeError,
        # so a narrow except would miss it and crash the whole study.
        err = str(e)
        cuda_keywords = ("CUDA", "cuda", "out of memory", "AcceleratorError", "accelerator")
        if any(kw in err for kw in cuda_keywords):
            raise optuna.TrialPruned()
        raise  # non-CUDA exceptions: re-raise so bugs aren't silently swallowed

    finally:
        # Always runs — even after CUDA errors where the context may be corrupted.
        # All cleanup ops are individually wrapped so one failure can't block the rest.
        if model is not None:
            try:
                del model
            except Exception:
                pass
        gc.collect()
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

# ── Create / resume study ─────────────────────────────────────────────────────
study = optuna.create_study(
    direction="minimize",
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=REPORT_EVERY,
        interval_steps=1,
    ),
)

# ── Diagnose existing DB state before running ─────────────────────────────────
n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
n_failed   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)
print(f"Study '{STUDY_NAME}' loaded — existing trials: {n_complete} complete, {n_pruned} pruned, {n_failed} failed")
if n_complete == 0 and (n_pruned + n_failed) > 0:
    print("WARNING: No completed trials in DB. Previous CUDA errors may have poisoned the study.")
    print("Recommended: delete the DB file and restart the runtime before running again.")
    print(f"  DB: {DB_PATH}")
print(f"Running {N_TRIALS} more trials...\n")

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

# ── Report results ────────────────────────────────────────────────────────────
n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
n_pruned   = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)

print(f"\nCompleted: {n_complete}  |  Pruned: {n_pruned}")

if n_complete == 0:
    print("\nNo trials completed successfully.")
    print("All trials were pruned — most likely the CUDA context is corrupted from a")
    print("previous OOM. Steps to recover:")
    print("  1. Runtime > Restart runtime  (resets the CUDA context)")
    print("  2. Delete the DB to start fresh:")
    print(f"       import os; os.remove('{DB_PATH}')")
    print("  3. Re-run all cells from the top")
else:
    best = study.best_trial
    print("\n" + "═" * 60)
    print("  BEST TRIAL")
    print("═" * 60)
    print(f"  Trial #     : {best.number}")
    print(f"  val loss    : {best.value:.4f}  (PPL {math.exp(best.value):.1f})")
    print(f"  n_params    : {best.user_attrs.get('n_params_M', '?')} M")
    print(f"  n_heads     : {best.user_attrs.get('n_heads', '?')} (head_dim=32)")
    print("  params      :")
    for k, v in best.params.items():
        print(f"    {k:<15} {v}")
    print(f"\nDB : {DB_PATH}")


In [ ]:
n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
if n_complete == 0:
    print("No completed trials to display. See the run-study cell for recovery steps.")
else:
    rows = []
    for t in study.trials:
        if t.state != optuna.trial.TrialState.COMPLETE:
            continue
        row = {
            "trial":      t.number,
            "val_loss":   round(t.value, 4),
            "val_ppl":    round(math.exp(t.value), 1),
            "n_params_M": t.user_attrs.get("n_params_M", None),
            "n_heads":    t.user_attrs.get("n_heads", None),
        }
        row.update(t.params)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("val_loss").reset_index(drop=True)
    cols = ["trial", "val_loss", "val_ppl", "n_params_M", "d_model", "n_heads",
            "n_layers", "d_ff", "lr", "weight_decay"]
    df = df[[c for c in cols if c in df.columns]]
    pd.set_option("display.float_format", "{:.4g}".format)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 200)
    print(df.to_string(index=False))
    n_pruned = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    print(f"\nCompleted: {n_complete}  |  Pruned: {n_pruned}")


In [ ]:
n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
if n_complete < 2:
    print(f"Need at least 2 completed trials for plots (have {n_complete}). Skipping.")
else:
    import plotly.io as pio
    pio.renderers.default = "notebook"
    optuna.visualization.plot_optimization_history(study).show()
    optuna.visualization.plot_param_importances(study).show()
    optuna.visualization.plot_parallel_coordinate(study).show()
    optuna.visualization.plot_slice(study).show()
    optuna.visualization.plot_contour(study, params=["lr", "d_ff"]).show()


In [ ]:
n_complete = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
if n_complete == 0:
    print("No completed trials. Cannot generate config.")
else:
    bp  = study.best_params
    bua = study.best_trial.user_attrs
    print(f"""
[model]
vocab_size        = {VOCAB_SIZE}
context_length    = {CONTEXT_LENGTH}
d_model           = {bp['d_model']}
n_heads           = {bua.get('n_heads', bp['d_model'] // 32)}
n_layers          = {bp['n_layers']}
d_ff              = {bp['d_ff']}
dropout           = {DROPOUT}
activation        = \"{ACTIVATION}\"
norm_position     = \"{NORM_POS}\"
weight_tying      = {str(WEIGHT_TYING).lower()}
bias              = {str(BIAS).lower()}
qk_norm           = {str(QK_NORM).lower()}
positional_encoding = \"{POS_ENC}\"
rope_theta        = 10000.0

[training]
learning_rate     = {bp['lr']:.2e}
weight_decay      = {bp['weight_decay']:.3f}
optimizer         = \"adamw\"
betas             = [0.9, 0.95]
lr_schedule       = \"cosine\"
min_lr_ratio      = 0.1
gradient_clip_norm = 1.0

# approx {bua.get('n_params_M', '?')} M parameters
# Optuna trial #{study.best_trial.number}  val_loss={study.best_value:.4f}
""")


## Optuna Dashboard (browser)

**Local** — run from the repo root:

```bash
optuna-dashboard sqlite:///notebooks/tiny_llm_optuna_search.db --host 127.0.0.1 --port 8080
```

Then open: http://127.0.0.1:8080

**Colab** — the DB is persisted on Drive at `MyDrive/tiny_llm/optuna/tiny_llm_optuna_search.db`.
You can download it and run the dashboard locally, or expose it from Colab via ngrok:

```python
# Run in a Colab code cell:
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess
db_url = "sqlite:////content/drive/MyDrive/tiny_llm/optuna/tiny_llm_optuna_search.db"
subprocess.Popen(["optuna-dashboard", db_url, "--host", "0.0.0.0", "--port", "8080"])
print("Dashboard:", ngrok.connect(8080))
```

**What to look for:**
- **History** — objective should trend downward as TPE learns. Flat/rising = too few trials.
- **Param Importance** — if `lr` dominates, run a finer log-range sweep in a follow-up study.
- **Parallel Coordinate** — filter to low-loss trials; see which `d_model` + `n_layers` + `d_ff` combos cluster there.
- **Slice** — individual parameter effects; confirms the `lr` optimum direction.
- **Contour (`lr` vs `d_ff`)** — the most informative 2-D view given prior ablation findings.
